In [73]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,END
from langgraph.graph.message import add_messages
from langchain.tools import tool
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict,Annotated
from langgraph.types import interrupt,Command

In [74]:
load_dotenv()

True

In [75]:
class MyState(TypedDict):
    receiver:str
    draft:str
    approval:str
    review_comments : str
    messages: Annotated[list,add_messages]

In [76]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [77]:
@tool
def send_email(draft:str)->dict:
    """This tool sends email"""
    print(draft)
    print("Email sent!")



In [78]:
llm_with_tools  = llm.bind_tools([send_email])

In [90]:
def llm_node(state:MyState)->dict:
    system = SystemMessage(content="You are a Helpful assitent.")
    response = llm_with_tools.invoke([system]+state["messages"])
    return {'messages':[response]}

def generate_email_draft(state:MyState)->dict:
    mes = state['messages']
    if state.get('approval') is None:
        data = HumanMessage(content=f"reciver name is {state['receiver']}")
        draft = llm_with_tools.invoke(mes+[data])
        print('this is draft')
        return {'draft':draft.content,'messages':[draft]}
    else:
        comments = HumanMessage(content=state['review_comments'])
        draft = llm_with_tools.invoke([comments]+mes)
        return {'draft':draft.content}


send_email_node = ToolNode([send_email])

def review_email(state: MyState) -> dict:
    descion = interrupt("What you wanna do nigga(yes/no/edit)")
    
    if descion['result'] == 'edit':
         print("this is descion",descion)
         return {'approval':descion['result'],'review_comments':descion['comments']}
    return {'approval':descion['result']}
    



In [80]:
# define routing function
def after_review(state:MyState):
    last = state["messages"][-1]

    if getattr(last,"tool_calls") and state['approval'] == 'yes':
        return "send_email"
    elif state['approval'] == 'edit':
        return "draft"
    return "end"


In [81]:
flow = StateGraph(MyState)
flow.add_node("llm_node",llm_node)
flow.add_node("draft",generate_email_draft)
flow.add_node("review",review_email)
flow.add_node("send",send_email_node)

flow.set_entry_point("llm_node")

In [82]:
flow.add_edge("llm_node","draft")
flow.add_edge("draft","review")
flow.add_edge("review","send")
flow.add_conditional_edges("review",
                           after_review,
                           {
                               'send_email':'send',
                               'draft':'draft',
                               'end':END
                           })                           

In [83]:
memory = MemorySaver()

In [84]:
app = flow.compile(checkpointer=memory)
config1 = {'configurable':{'thread_id':'Narsimha_thread'}}
config2 = {'configurable':{'thread_id':'Anand_thread'}}

In [91]:
result = app.invoke({'messages':[HumanMessage(content="write a very short email to my friend.")],'receiver':'Narsimha'},
                    config = config1)

print(result['messages'][-1].content)

This is state: {'receiver': 'Narsimha', 'draft': 'Subject: Quick check‑in  \n\nHey [Friend’s Name],\n\nHow’s everything going? By the way, how’s Mia doing? Hope you both are well!\n\nTalk soon,  \n[Your Name]', 'approval': 'yes', 'review_comments': 'ask about his gf Mia.', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='0f5ba161-7edd-49c7-a709-7ce0682b7cb4'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say I’m thinking of you. Hope all’s great on your end—let’s catch up soon!\n\nTake care,  \n[Your Name]', additional_kwargs={'reasoning_content': 'User wants a very short email to their friend. Need to produce email content. Probably just text. No need to send. Should ask for any specifics? Probably just give a short email.'}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 137, 'total_tokens': 233, 'completion_time': 0.2002

In [92]:
print(result['__interrupt__'][-1].value)


What you wanna do nigga(yes/no/edit)


Resuming Graph execution after interrupt

In [93]:
result = app.invoke(Command(resume={'result':'edit','comments':'ask about his gf Mia.'}),
           config=config1)

print(result['messages'][-1].content)

this is descion {'result': 'edit', 'comments': 'ask about his gf Mia.'}
This is state: {'receiver': 'Narsimha', 'draft': '**Subject:** Quick hello  \n\nHey [Friend’s Name],\n\nJust wanted to say hi and see how you’re doing. Let’s catch up soon!\n\nBest,  \n[Your Name]', 'approval': 'edit', 'review_comments': 'ask about his gf Mia.', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='0f5ba161-7edd-49c7-a709-7ce0682b7cb4'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say I’m thinking of you. Hope all’s great on your end—let’s catch up soon!\n\nTake care,  \n[Your Name]', additional_kwargs={'reasoning_content': 'User wants a very short email to their friend. Need to produce email content. Probably just text. No need to send. Should ask for any specifics? Probably just give a short email.'}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt

In [94]:
result = app.invoke(Command(resume={'result':'yes'}),
           config=config1)
print(app.get_state(config=config1))

StateSnapshot(values={'receiver': 'Narsimha', 'draft': '**Subject:** Quick hello  \n\nHey [Friend’s Name],\n\nJust wanted to say hi and see how you’re doing. Let’s catch up soon!\n\nBest,  \n[Your Name]', 'approval': 'yes', 'review_comments': 'ask about his gf Mia.', 'messages': [HumanMessage(content='write a very short email to my friend.', additional_kwargs={}, response_metadata={}, id='0f5ba161-7edd-49c7-a709-7ce0682b7cb4'), AIMessage(content='Subject: Quick Hello  \n\nHey [Friend’s Name],\n\nJust wanted to drop a quick note to say I’m thinking of you. Hope all’s great on your end—let’s catch up soon!\n\nTake care,  \n[Your Name]', additional_kwargs={'reasoning_content': 'User wants a very short email to their friend. Need to produce email content. Probably just text. No need to send. Should ask for any specifics? Probably just give a short email.'}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 137, 'total_tokens': 233, 'completion_time': 0.200282576,

Time Travel